In [ ]:
import torch

# ตรวจสอบและใช้งาน GPU ของชิป M4
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"รันโมเดลบน: {device}")

In [ ]:
import numpy as np
from scipy.io import loadmat

ai_model = 'cnn'
snr = 0
scenario = 'O1'
frequency = 140
antennas = 64

path = f'../DeepMIMO/DeepMIMO/DeepMIMO_dataset/SNR{snr}dB_{scenario}_{frequency}_Ant{antennas}/'
d1 = loadmat(path+'channel1.mat')['a']
d2 = loadmat(path+'channel2.mat')['b']
d3 = loadmat(path+'channel3.mat')['c']

data = np.concatenate((d1, d2, d3), axis=2).transpose(2, 0, 1)

d_r = data.real.reshape(-1, 1, 64, 32)
d_i = data.imag.reshape(-1, 1, 64, 32)

X = np.concatenate((d_r, d_i), axis=1)
mean = np.mean(X, axis=0)
std = np.std(X, axis=0)
X = (X - mean) / (std + 1e-8) # 1e-8 prevents division by zero

print("Data normalization complete. Features now have Mean ≈ 0 and Std ≈ 1.")

# Load Label (One-hot encoding)
y = loadmat(path+'DLCB_output.mat')['onehot_label']
se_data = loadmat(path+'rate_ave.mat')['DL_output']

print(f"Input shape: {X.shape}")
print(f"Label shape: {y.shape}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

split_idx = int(len(X) * 0.7)

# Training set: The first 70% of the user path
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

se_test = se_data[split_idx:]

print(f"Data Split Complete:")
print(f" - Training samples: {len(X_train)} (First 70%)")
print(f" - Testing samples:  {len(X_test)} (Last 30% - Future Path)")
print(f" - SE test shape: {se_test.shape}")

train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
test_ds = TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).float())

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random

def apply_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
        
    print(f"Global environment locked with seed: {seed}")

apply_global_seed(42)

class BeamPredictionCNN(nn.Module):
    def __init__(self, n_beams, dropout_rate=0.2):
        super(BeamPredictionCNN, self).__init__()
        
        self.cnn = nn.Sequential(
            # Feature Extraction
            nn.Conv2d(in_channels=2, out_channels=16, kernel_size=3, padding=1, bias=False), # Detects basic patterns.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(4, 2)), # Shrinks Height by 4, Width by 2.
            nn.Dropout(dropout_rate),
            
            # Deep Feature Extraction
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1, bias=False), # Detects complex MIMO patterns.
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=4), # Shrinks Height by 4, Width by 4.
            nn.Dropout(dropout_rate)
        )
        
        self.linear = nn.Sequential(
            nn.Linear(512, 164), # Dense Layer 1: 512 Nodes -> 164 Nodes.
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(164, n_beams) # 164 Nodes -> 64 Beams (Logits for CrossEntropyLoss).
        )

    def forward(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1) # Flatten for Linear Layer
        x = self.linear(x)
        return x

In [ ]:
model = BeamPredictionCNN(n_beams=64, dropout_rate=0.2).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
criterion = nn.CrossEntropyLoss()

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Trainable Params: {params}')

epochs = 100
best_loss = float('inf')
train_losses = []
save_path = './best_models/'

In [ ]:
for epoch in range(epochs):
    epoch_loss = 0.0
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        # Use torch.max to find the best index of beam for CrossEntropyLoss
        loss = criterion(outputs, torch.max(labels, 1)[1])
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_epoch_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_epoch_loss)
        
    scheduler.step(avg_epoch_loss)
        
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_epoch_loss:.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        torch.save(model.state_dict(), save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth')
        print(f"--> Saved better model at Epoch {epoch+1} with Loss: {best_loss:.4f}")

        

In [ ]:
from datetime import datetime as dt
import evaluate as ev
import visualizer as vis

model.load_state_dict(torch.load(
    save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth'))

eval_datetime = dt.now().strftime("%Y-%m-%d %H:%M:%S")

ds_config = {
    'snr': snr,
    'scenario': scenario,
    'frequency': frequency,
    'antennas': antennas

}

mimo_results = ev.evaluate_performance(
    model,
    test_loader,
    device,
    criterion,
    ai_model,
    ds_config,
    eval_datetime
)

# vis.plot_training_loss(train_losses, mimo_results, ds_config)

vis.plot_confusion_matrix(
    mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)

vis.plot_beam_tracking(
    mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)

# 1. เตรียมข้อมูลทั้ง 5 ส่วน (สมมติว่าเป็นข้อมูลของ 200 Users)
# user_indices: ลำดับของผู้ใช้งาน (0 ถึง 199)
current_user_indices = np.arange(200)

# optimal_se: ลิสต์หรืออาเรย์ของค่า SE ที่ดีที่สุด (เส้นสีเขียว)
current_optimal_se = np.random.uniform(10, 20, 200)

# predicted_se: ลิสต์หรืออาเรย์ของค่า SE ที่ AI ทำได้จริง (เส้นสีแดง)
current_predicted_se = current_optimal_se - np.random.uniform(0, 2, 200)

preds_array = np.array(mimo_results['all_preds'])
actuals_array = np.array(mimo_results['all_actuals'])

num_users = len(preds_array)
user_indices = np.arange(num_users)

predicted_se = se_test[user_indices, preds_array]
optimal_se = se_test[user_indices, actuals_array]

plot_limit = 200

vis.plot_se_tracking(
    user_indices=user_indices[:plot_limit],
    optimal_se=optimal_se[:plot_limit],
    predicted_se=predicted_se[:plot_limit],
    model_name=ai_model,
    ds_config=ds_config,
    eval_datetime=eval_datetime
)

vis.save_se_summary_to_csv(
    mimo_results=mimo_results,
    se_test=se_test,
    model_name=ai_model,
    ds_config=ds_config,
    eval_datetime=eval_datetime
)

In [ ]:
import torch

model.eval()

dummy_input = torch.randn(1, 2, 64, 32).to(device) 

onnx_file_path = "low_level_cnn.onnx"

torch.onnx.export(
    model,                      
    dummy_input,                
    onnx_file_path,             
    export_params=True,         
    opset_version=14,
    do_constant_folding=True,   
    input_names=['input'],      
    output_names=['output'],    
    
    dynamic_axes={
        'input': {0: 'batch_size'}, 
        'output': {0: 'batch_size'}
    }
)

print(f"Model successfully exported to {onnx_file_path}!")

In [ ]:
from torchview import draw_graph
import torch

model_cnn_graph = draw_graph(model, input_size=(1, 2, 64, 32), depth=2, expand_nested=False)
model_cnn_graph.visual_graph.render("cnn_architecture", format="png")